### Introduction to Data Ingestion

In [30]:
import os
from typing import List,Dict,Any
import pandas as pd

In [31]:
from langchain_core.documents import Document

### Understanding the Document Structure in LangChain

In [32]:
 # Creating the sample document to be ingested
from importlib import metadata


doc = Document(
    page_content=(
        "PostgreSQL with the pgvector extension allows developers to perform "
        "vector similarity search directly alongside traditional relational data. "
        "This eliminates the need for a separate vector database..."
    ),
    metadata={
        "source_file": "architecture_guidelines_2026.pdf",
        "document_title": "Enterprise Vector DB Selection",
        "author": "Engineering Team",
        "access_level": "internal", 
        "category": "technical_docs",
        "creation_date": "2026-09-10",
        "chunk_index": 1
    }
)
print("DOCUMENT STRUCTURE")
print(f"content:{doc.page_content}")
print(f"Metadata:{doc.metadata}")

DOCUMENT STRUCTURE
content:PostgreSQL with the pgvector extension allows developers to perform vector similarity search directly alongside traditional relational data. This eliminates the need for a separate vector database...
Metadata:{'source_file': 'architecture_guidelines_2026.pdf', 'document_title': 'Enterprise Vector DB Selection', 'author': 'Engineering Team', 'access_level': 'internal', 'category': 'technical_docs', 'creation_date': '2026-09-10', 'chunk_index': 1}


In [33]:
type(doc)

langchain_core.documents.base.Document

### Reading a Text File

In [34]:
## create a simple txt file
os.makedirs("data/text_files",exist_ok=True)

In [35]:

sample_texts={
    "data/text_files/python_intro.txt":"""Python is a popular programming language known for being clear and easy to read, almost like writing plain English.

Instead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.

Because of its simplicity and vast collection of free tools, it is widely used for:

* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.
* **Web Development:** Powering the back-end logic of websites and web apps.
* **Automation:** Writing small scripts to take care of repetitive daily tasks."""
}
for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)

print("sample file path created")

sample file path created


### TextLoader-read Single File

In [36]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
docs=loader.load()
# 1. Print the entire metadata dictionary
print(docs[0].metadata)

# 2. Print just the source file path
print(docs[0].metadata['source'])

# 3. Print the text content only
print(docs[0].page_content)


{'source': 'data/text_files/python_intro.txt'}
data/text_files/python_intro.txt
Python is a popular programming language known for being clear and easy to read, almost like writing plain English.

Instead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.

Because of its simplicity and vast collection of free tools, it is widely used for:

* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.
* **Web Development:** Powering the back-end logic of websites and web apps.
* **Automation:** Writing small scripts to take care of repetitive daily tasks.


### Directoryloader-Multiple Text Files

In [37]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)
documents=dir_loader.load()

print(f"Loaded {len(documents)} documents")
for i ,doc in  enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f" Source: {doc.metadata["source"]}")
    print(f" Length: {len(doc.page_content)} charcters")


100%|██████████| 2/2 [00:00<00:00, 2003.49it/s]

Loaded 2 documents

Document 1:
 Source: data\text_files\ML intro.txt
 Length: 2174 charcters

Document 2:
 Source: data\text_files\python_intro.txt
 Length: 701 charcters


### Text Splitting Strategies

In [40]:
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print(docs)


[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a popular programming language known for being clear and easy to read, almost like writing plain English.\n\nInstead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.\n\nBecause of its simplicity and vast collection of free tools, it is widely used for:\n\n* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.\n* **Web Development:** Powering the back-end logic of websites and web apps.\n* **Automation:** Writing small scripts to take care of repetitive daily tasks.')]


In [53]:
### Method-1 Character Text Splitter
text=docs[0].page_content
text

'Python is a popular programming language known for being clear and easy to read, almost like writing plain English.\n\nInstead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.\n\nBecause of its simplicity and vast collection of free tools, it is widely used for:\n\n* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.\n* **Web Development:** Powering the back-end logic of websites and web apps.\n* **Automation:** Writing small scripts to take care of repetitive daily tasks.'

In [49]:
print("Character Text Splitter")
char_splitter=CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=20,
    length_function=len
)

char_chunks=char_splitter.split_text(text)
print(char_chunks[0])
print(char_chunks[1])

Created a chunk of size 115, which is longer than the specified 100
Created a chunk of size 239, which is longer than the specified 100


Character Text Splitter
Python is a popular programming language known for being clear and easy to read, almost like writing plain English.
Instead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.


In [60]:
### Method-2 Recursive Character Text Splitter(Recommended)
print("\n Recursive Character Text Splitter")
recursive_splitter=RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
recursive_chunks=recursive_splitter.split_text(text)
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("-----------------")
print(recursive_chunks[2])
print("-----------------")
print(recursive_chunks[3])
print("-----------------")


 Recursive Character Text Splitter
Python is a popular programming language known for being clear and easy to read, almost like writing plain English.

Instead of getting bogged down by complicated syntax, punctuation, or complex setup
-----------------
or complex setup rules, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.

Because of its simplicity
-----------------
of its simplicity and vast collection of free tools, it is widely used for:

* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.
* **Web
-----------------
models.
* **Web Development:** Powering the back-end logic of websites and web apps.
* **Automation:** Writing small scripts to take care of repetitive daily tasks.
-----------------


In [66]:
### Method-3 Token Based Splitting
print("Token based splitting")
token_splitter=TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
token_chunks=token_splitter.split_text(text)
print(f"created {len(token_chunks)} chunks")
print(token_chunks[0])
print("--------------")
print(token_chunks[1])
print("--------------")
print(token_chunks[2])
print("--------------")
print(token_chunks[3])

Token based splitting
created 4 chunks
Python is a popular programming language known for being clear and easy to read, almost like writing plain English.

Instead of getting bogged down by complicated syntax, punctuation, or complex setup rules, it lets you solve problems with fewer lines of
--------------
, it lets you solve problems with fewer lines of code. It runs your instructions directly line by line, making it fast to test ideas and fix mistakes.

Because of its simplicity and vast collection of free tools, it is widely used for:
--------------
 of free tools, it is widely used for:

* **Artificial Intelligence & Data Science:** Analyzing numbers and building machine learning models.
* **Web Development:** Powering the back-end logic of websites and web apps
--------------
 the back-end logic of websites and web apps.
* **Automation:** Writing small scripts to take care of repetitive daily tasks.
